# How UniChart Handles Data &mdash; A Walkthrough

This tutorial explains UniChart's **data model**: where your data actually lives, what a
"dataset" really is, and how every read, write, filter, and selection follows from one
central design choice. It is the foundation for the other notebooks:

- **`adding_parameters_demo.ipynb`** &mdash; the full catalog of ways to add/compute columns;
- **`delta_demo.ipynb`** &mdash; comparing datasets and interpolating between them;
- **`UnichartNotebook_Tutorial.ipynb`** &mdash; plotting and styling.

Here we deliberately stay off plotting and focus on the data itself. Each behavioral claim
below is backed by a runnable cell &mdash; cells end in a `PASS` print where the behavior can be
asserted outright.

## 1. The one big idea: a single combined frame

When you load several datasets, UniChart does **not** keep a separate DataFrame per set.
Instead it keeps:

- **one** combined `pandas.DataFrame` (`nb.df`), with every set's rows stacked together and an
  internal **`_SET_ID`** column tagging which set each row belongs to;
- a list of lightweight **`Dataset` façades** (`nb.sets`) &mdash; each one is just a *view* onto the
  rows of the frame whose `_SET_ID` matches it, plus that set's styling/state.

Three consequences flow from this, and the rest of the notebook is really just these three:

1. **`ds.df` is a fresh copy** computed on each access (a masked slice of the frame), so reads
   are safe but writes through it don't persist (Section 6).
2. **Storage is the union, views are owned.** Because all sets share one frame, a column
   added for one set physically exists (NaN-filled) for every other set. But each set's
   `df`/`columns` shows only the columns it **owns** — the ones it was loaded with or has
   been given values for — so other sets' all-NaN "phantom" columns stay out of view
   (Section 6).
3. Filtering and selection are **non-destructive flags/masks** layered on top of the frame,
   not edits to it (Sections 4&ndash;5).

In [ ]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook
from unichart import _SET_ID_COL   # the internal tag column name ('_SET_ID')

nb = UnichartNotebook()
nb.load({'T': [0, 1, 2, 3], 'RPM': [800, 1200, 1600, 2000], 'TEMP': [40, 55, 72, 90]}, title='Run-A')
nb.load({'T': [0, 1, 2],    'RPM': [820, 1250, 1680],       'TEMP': [42, 58, 77]},      title='Run-B')

print('nb.sets  ->', nb.sets)                 # list of Dataset facades
print('types    ->', [type(d).__name__ for d in nb.sets])
print()
print('nb.df (the ONE combined frame, _SET_ID tags each row):')
print(nb.df)

assert _SET_ID_COL in nb.df.columns               # the tag column is real
assert len(nb.df) == 4 + 3                         # both sets' rows stacked together
print('\nPASS - all sets live in one frame, tagged by _SET_ID.')

UniChart Notebook Environment Initialized.
Loaded Set 0: Run-A
Loaded Set 1: Run-B
nb.sets  -> [<unichart.Dataset object at 0x7a5aad1a9400>, <unichart.Dataset object at 0x7a5aacfcaf00>]
types    -> ['Dataset', 'Dataset']

nb.df (the ONE combined frame, _SET_ID tags each row):
   T   RPM  TEMP  _SET_ID
0  0   800    40        0
1  1  1200    55        0
2  2  1600    72        0
3  3  2000    90        0
4  0   820    42        1
5  1  1250    58        1
6  2  1680    77        1

PASS - all sets live in one frame, tagged by _SET_ID.


### `ds.df` is a fresh copy each time

Accessing `ds.df` builds a new masked slice on the spot &mdash; so two accesses are *not* the same
object. That is exactly why writing through it won't stick (we return to this in Section 6).

In [ ]:
ds = nb.sets[0]
print('ds.df is ds.df ?', ds.df is ds.df)     # False: a new copy each access
assert ds.df is not ds.df
print('PASS - ds.df returns a fresh copy on every read.')

ds.df is ds.df ? False
PASS - ds.df returns a fresh copy on every read.


### Columns: union storage, owned views

Because the sets share one frame, adding a column to **one** set makes that column physically
exist for **every** set &mdash; NaN-filled where it has no value. But a set's `df` doesn't *show*
those foreign all-NaN "phantom" columns: each set tracks the columns it **owns**, and only those
appear in `ds.df` / `ds.columns`. Ownership follows the data &mdash; give a set values for a column
(through any write API) and it claims that column. (We add columns properly in Section 6; here
we just observe the consequence.)

In [ ]:
nb.sets[0]['NOTE'] = 'hello'        # write a column onto Run-A only (Section 6 covers writes)

print('Run-A columns:', list(nb.sets[0].df.columns))
print('Run-B columns:', list(nb.sets[1].df.columns), ' <- no NOTE: Run-B does not own it')

assert 'NOTE' in nb.df.columns                    # union storage: the column exists frame-wide
assert 'NOTE' not in nb.sets[1].df.columns        # owned view: Run-B does not show it
assert nb.df.loc[nb.df[_SET_ID_COL] == nb.sets[1]._set_id, 'NOTE'].isna().all()   # stored as NaN

nb.set_column(1, 'NOTE', 'hi from B')             # giving Run-B values claims the column...
assert 'NOTE' in nb.sets[1].df.columns            # ...so now Run-B's view shows it
print('PASS - union storage, owned views; ownership follows the data.')

Run-A columns: ['T', 'RPM', 'TEMP', 'NOTE']
Run-B columns: ['T', 'RPM', 'TEMP', 'NOTE']  <- NOTE appears here too, as NaN
PASS - one frame => columns are the union across all sets.


## 2. Loading data

`nb.load(source, ...)` is the front door. A `source` can be a **DataFrame, dict, filepath**
(`.csv/.tsv/.txt/.xlsx/.xls/.json/.parquet`), or **numpy array**. Pass a **list** of sources to
load several at once. Two behaviors are worth knowing:

- **`combined=True`** concatenates a list of sources into a *single* set instead of one each;
- a frame carrying a **`SETNUMBER`** or **`INDEX`** column is automatically **split into one set
  per distinct value** &mdash; handy for long-format data where many runs share one table.

In [ ]:
# (a) A list of sources -> one set each.
nb_a = UnichartNotebook()
nb_a.load([{'X': [1, 2, 3], 'Y': [1, 4, 9]},
           {'X': [1, 2, 3], 'Y': [1, 8, 27]}], title=None)
print('(a) list  -> sets:', len(nb_a.sets))
assert len(nb_a.sets) == 2

# (b) Same list, combined=True -> a single set.
nb_b = UnichartNotebook()
nb_b.load([{'X': [1, 2, 3], 'Y': [1, 4, 9]},
           {'X': [1, 2, 3], 'Y': [1, 8, 27]}], combined=True, title='Merged')
print('(b) combined=True -> sets:', len(nb_b.sets), '| rows:', len(nb_b.sets[0].df))
assert len(nb_b.sets) == 1 and len(nb_b.sets[0].df) == 6

# (c) Long-format frame split automatically by a SETNUMBER column.
long_df = pd.DataFrame({'SETNUMBER': [0, 0, 1, 1, 2, 2],
                        'X': [1, 2, 1, 2, 1, 2],
                        'Y': [10, 20, 11, 22, 12, 24]})
nb_c = UnichartNotebook()
nb_c.load(long_df)
print('(c) split by SETNUMBER -> sets:', len(nb_c.sets))
assert len(nb_c.sets) == 3
print('PASS - list, combined, and split-by-column loading all behave as documented.')

UniChart Notebook Environment Initialized.
Loaded Set 0: Untitled
Loaded Set 1: Untitled
(a) list  -> sets: 2
UniChart Notebook Environment Initialized.
Loaded Set 0: Merged
(b) combined=True -> sets: 1 | rows: 6
UniChart Notebook Environment Initialized.
Loaded Set 0: Dataset
Loaded Set 1: Dataset
Loaded Set 2: Dataset
(c) split by SETNUMBER -> sets: 3
PASS - list, combined, and split-by-column loading all behave as documented.


## 3. Inspecting what you have

Three built-ins answer "what's loaded and what's in it?" without digging into the frame:

- **`list_sets()`** &mdash; a table of every set: index, title, selected?, shape, active query;
- **`list_parms()`** &mdash; the parameters (columns) available (optionally filtered/searched);
- **`summary(cols=...)`** &mdash; per-set count/min/mean/max/std for the chosen columns (selected sets).

In [ ]:
nb.list_sets()
print()
nb.list_parms()


Loaded Datasets:
Set    Title  Selected  Shape  Query?  
---------------------------------------
Set 0  Run-A  ✓         4 x 4  None    
Set 1  Run-B  ✓         3 x 4  None    

Found 4 parameters in active datasets:
  - NOTE                      : No description available.
  - RPM                       : No description available.
  - T                         : No description available.
  - TEMP                      : No description available.


['NOTE', 'RPM', 'T', 'TEMP']

In [ ]:
nb.select('all')                       # summary works over the *selected* sets (Section 4)
nb.summary(cols=['RPM', 'TEMP'])       # returns a tidy stats DataFrame

,Set,Title,Query,Variable,Count,Min,Mean,Max,Std
0,0,Run-A,-,RPM,4,800,1400.00,2000,516.397779
1,0,Run-A,-,TEMP,4,40,64.25,90,21.577380
2,1,Run-B,-,RPM,3,820,1250.00,1680,430.000000
3,1,Run-B,-,TEMP,3,42,59.00,77,17.521415


## 4. Selecting which sets are "active"

Every set carries a boolean **`select`** flag. Most downstream operations &mdash; `plot()`,
`table()`, `summary()`, the column stats &mdash; act on the **selected** sets only. Selection is a
flag, never a deletion:

- **`select(...)`** &mdash; turn the given set(s) on and everything else off;
- **`omit(...)`** / **`restore(...)`** &mdash; turn specific sets off / back on;
- **`selected()`** &mdash; the list of currently-active `Dataset`s.

Selectors are by **index** (an int, a list of ints, a `Dataset`, or `'all'`) &mdash; titles are not
selectors. (Negative indices count from the end.)

In [ ]:
nb.select(0)                                  # only Run-A active
print('after select(0):', [d.title for d in nb.selected()])
assert [d.title for d in nb.selected()] == ['Run-A']

nb.restore(1)                                 # bring Run-B back too
print('after restore(1):', [d.title for d in nb.selected()])

nb.omit(0)                                    # drop Run-A from the active set
print('after omit(0):', [d.title for d in nb.selected()])
assert [d.title for d in nb.selected()] == ['Run-B']

nb.select('all')                              # back to everything
assert len(nb.selected()) == 2
print('PASS - selection is a non-destructive on/off flag that drives downstream ops.')

after select(0): ['Run-A']
after restore(1): ['Run-A', 'Run-B']
after omit(0): ['Run-B']
PASS - selection is a non-destructive on/off flag that drives downstream ops.


## 5. Querying: non-destructive row filtering

`query(sets, 'expr')` applies a pandas query string to chosen sets. Crucially it is **a mask,
not a delete**: `ds.df` (and anything reading it) sees only the matching rows, but the
underlying frame is untouched, so a later `query(sets, None)` brings every row back. (If a
query leaves a set empty, that set is simply switched off.)

In [ ]:
print('Run-A rows, unfiltered:', len(nb.sets[0].df))

nb.query(0, 'RPM > 1000')                      # mask Run-A to its higher-RPM rows
print("after query 'RPM > 1000':", len(nb.sets[0].df), 'rows visible')
print('  but the raw rows are still in the frame:', len(nb.sets[0]._raw_df()))

assert len(nb.sets[0].df) == 3          # masked view
assert len(nb.sets[0]._raw_df()) == 4   # frame still holds every row

nb.query(0, None)                              # clear the query -> all rows return
print('after clearing the query:', len(nb.sets[0].df), 'rows visible again')
assert len(nb.sets[0].df) == 4
print('PASS - query masks rows non-destructively; clearing restores them.')

Run-A rows, unfiltered: 4
after query 'RPM > 1000': 3 rows visible
  but the raw rows are still in the frame: 4
after clearing the query: 4 rows visible again
PASS - query masks rows non-destructively; clearing restores them.


## 6. Reading vs. writing columns

**Reading** is easy: `ds['col']` or `ds.df` (both honor the active query).

**Writing** has one rule that follows directly from Section 1 &mdash; *`ds.df` is a copy*:

- &#9989; `ds['col'] = ...` writes back into the shared frame for that set (it **persists**);
- &#10060; `ds.df['col'] = ...` mutates a throwaway copy and is **silently lost**.

There are also notebook-level writers &mdash; **`add_column(name, fn)`** (one formula across all
sets) and **`set_column(sets, col, vals)`** (a block into chosen sets). The full catalog, plus
computing a column *differently per set*, lives in **`adding_parameters_demo.ipynb`**; here we
just establish the persistence rule.

In [ ]:
ds = nb.sets[0]

ds['POWER'] = ds['RPM'] * ds['TEMP']     # CORRECT: persists into the frame for Run-A
ds.df['LOST'] = -1                        # WRONG: writes to a copy, evaporates

cols = list(nb.sets[0].df.columns)
print('Run-A columns now:', cols)
assert 'POWER' in cols, 'ds[...] = should persist'
assert 'LOST' not in cols, 'ds.df[...] = should NOT persist'
print('PASS - ds[col]= persists; ds.df[col]= is a copy and is lost.')

Run-A columns now: ['T', 'RPM', 'TEMP', 'NOTE', 'POWER']
PASS - ds[col]= persists; ds.df[col]= is a copy and is lost.


/tmp/ipykernel_139609/2342112025.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  ds.df['LOST'] = -1                        # WRONG: writes to a copy, evaporates


## 7. Derived sets live in the same frame

Operations like **`combine_sets(...)`** and **`delta(...)`** don't return loose DataFrames &mdash;
they **register a new set** in the very same combined frame, so it shows up in `nb.sets`,
obeys selection/query, and can be inspected like any other. Their new columns (e.g. a delta's
`DL_*`) join the shared frame's storage, but &mdash; per the ownership rule from Section 1 &mdash; they
stay out of the *original* sets' views. `delta()` is covered in depth in **`delta_demo.ipynb`**.

In [ ]:
before = len(nb.sets)
combo = nb.combine_sets([0, 1], title='A+B stacked')   # row-bind Run-A and Run-B into a new set
print('sets before:', before, '-> after:', len(nb.sets))
print('new set:', nb.sets[-1].index, nb.sets[-1].title, '| rows:', len(nb.sets[-1].df))

assert len(nb.sets) == before + 1
assert len(nb.sets[-1].df) == len(nb.sets[0]._raw_df()) + len(nb.sets[1]._raw_df())
print('PASS - derived sets are first-class members of the same frame.')

Loaded Set 2: A+B stacked (7 rows from 2 datasets)
sets before: 2 -> after: 3
new set: 2 A+B stacked | rows: 7
PASS - derived sets are first-class members of the same frame.


## 8. `index` vs. `_SET_ID` &mdash; two numbers, one (today) the same

You will see two identifiers for a set:

- **`ds.index`** &mdash; the set's **position in `nb.sets`**, and what every selector uses;
- **`ds._set_id`** &mdash; the **stable tag in the frame's `_SET_ID` column**, the join key that ties
  rows to their set.

Conceptually they play different roles (one is a list position, one is a storage key). In
practice they **coincide by construction**: sets are only ever *appended*, and both counters
advance together &mdash; there is no remove/reorder operation that could make them drift. So you
can address sets by their index and trust it, while understanding that `_SET_ID` is what
actually stitches the frame together.

In [ ]:
print('(index, _set_id, title):')
for d in nb.sets:
    print('  ', (d.index, d._set_id, d.title))

# index == set_id for every set (true by construction today)
assert all(d.index == d._set_id for d in nb.sets)
# and _SET_ID in the frame really does map rows to those sets
assert set(nb.df[_SET_ID_COL].unique()) == {d._set_id for d in nb.sets}
print('PASS - index and _SET_ID coincide; _SET_ID is the frame-level join key.')

(index, _set_id, title):
   (0, 0, 'Run-A')
   (1, 1, 'Run-B')
   (2, 2, 'A+B stacked')
PASS - index and _SET_ID coincide; _SET_ID is the frame-level join key.


---
### Summary &mdash; the model in one breath

- **One combined frame** (`nb.df`) holds every set's rows, tagged by **`_SET_ID`**; **`nb.sets`** are
  lightweight **views** (`Dataset`) onto it.
- **`ds.df` is a fresh copy** &mdash; great for reading, useless for writing. Persist with **`ds['col']=...`**.
- **Storage is the union, views are owned**: the shared frame NaN-fills every column across
  sets, but `ds.df` shows only the columns a set owns (ownership follows the data);
  **selection** and **query** are non-destructive flags/masks, not edits.
- **Derived sets** (`delta`, `combine_sets`) are registered into the same frame as full citizens.

From here: **`adding_parameters_demo.ipynb`** (computing columns, incl. per-set formulas),
**`delta_demo.ipynb`** (comparing/interpolating sets), and **`UnichartNotebook_Tutorial.ipynb`**
(plotting & styling).